# Question 1: Frequent Item Set Mining (Apriori)
FISM algorithm used: **Apriori**. Each recipe is a *transaction* and each ingredient is an *item*.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.makedirs('outputs', exist_ok=True)

# Load recipe -> ingredient list (each recipe is a set of unique ingredient names)
df = pd.read_csv('recipe_id_ingredient_name.csv').dropna()
df['ingredient_name'] = df['ingredient_name'].str.strip().str.lower()
df = df.drop_duplicates()
recipes = {rid: sorted(set(g)) for rid, g in df.groupby('recipe_id')['ingredient_name']}
print('Number of recipes:', len(recipes))
print('Number of unique ingredients:', df['ingredient_name'].nunique())
from itertools import combinations
from collections import Counter

N = len(recipes)
transactions = [frozenset(v) for v in recipes.values()]

Number of recipes: 9999
Number of unique ingredients: 13226


## 1(a) Apriori and complete list of item sets of size 1, 2 and 3
Steps:
1. **L1**: count every ingredient; keep those with count >= `MIN_COUNT`.
2. **C2 / L2**: candidate pairs are built only from frequent single ingredients; count them over all recipes.
3. **C3 / L3**: candidate triples are kept only if *all* their 2-subsets are in L2 (Apriori pruning, because support is anti-monotone); count them.
4. Support = count / N.

The size-1 list is complete (every ingredient). Size 2 and 3 lists contain item sets occurring in at least `MIN_COUNT` = 2 recipes (set `MIN_COUNT = 1` for every possible combination).

In [2]:
MIN_COUNT = 2

def apriori(transactions, min_count, max_k=3):
    levels = {}
    # Step 1: frequent 1-itemsets
    c1 = Counter(i for t in transactions for i in t)
    levels[1] = {(i,): c for i, c in c1.items()}          # complete list (count >= 1)
    f1 = {i for i, c in c1.items() if c >= min_count}
    # keep only frequent items inside each transaction (Apriori property)
    ftx = [sorted(i for i in t if i in f1) for t in transactions]

    # Step 2: candidate pairs -> count -> filter
    c2 = Counter(p for t in ftx for p in combinations(t, 2))
    levels[2] = {k: c for k, c in c2.items() if c >= min_count}
    f2 = set(levels[2])

    # Step 3: candidate triples, pruned: all 2-subsets must be frequent
    c3 = Counter()
    for t in ftx:
        for a, b, c in combinations(t, 3):
            if (a, b) in f2 and (a, c) in f2 and (b, c) in f2:
                c3[(a, b, c)] += 1
    levels[3] = {k: c for k, c in c3.items() if c >= min_count}
    return levels

levels = apriori(transactions, MIN_COUNT)

tables = {}
for k, d in levels.items():
    t = pd.DataFrame([{'itemset': ' | '.join(s), 'size': k, 'count': c, 'support': c / N}
                      for s, c in d.items()])
    tables[k] = t.sort_values(['count', 'itemset'], ascending=[False, True]).reset_index(drop=True)
    print(f'size {k}: {len(tables[k])} item sets')

size 1: 13226 item sets
size 2: 49176 item sets


size 3: 123195 item sets


### Save complete lists to CSV

In [3]:
for k in (1, 2, 3):
    tables[k].to_csv(f'outputs/Q1a_itemsets_size{k}.csv', index=False)
print('saved: Q1a_itemsets_size1.csv, Q1a_itemsets_size2.csv, Q1a_itemsets_size3.csv')

saved: Q1a_itemsets_size1.csv, Q1a_itemsets_size2.csv, Q1a_itemsets_size3.csv


## 1(b) Top 20 item sets of size 1, 2 and 3 (descending order of support)

In [4]:
for k in (1, 2, 3):
    top = tables[k].head(20)
    print(f'\n=== Top 20 item sets of size {k} ===')
    print(top.to_string())
    top.to_csv(f'outputs/Q1b_top20_size{k}.csv', index=False)


=== Top 20 item sets of size 1 ===
                  itemset  size  count   support
0               olive oil     1   2594  0.259426
1                  butter     1   1868  0.186819
2                   flour     1   1571  0.157116
3                    eggs     1   1472  0.147215
4                  garlic     1   1198  0.119812
5                   onion     1   1157  0.115712
6            caster sugar     1    976  0.097610
7                    milk     1    976  0.097610
8            garlic clove     1    900  0.090009
9     golden caster sugar     1    877  0.087709
10                  lemon     1    803  0.080308
11          vegetable oil     1    784  0.078408
12              coriander     1    774  0.077408
13           double cream     1    749  0.074907
14            icing sugar     1    743  0.074307
15                    egg     1    738  0.073807
16          spring onions     1    662  0.066207
17  garlic cloves crushed     1    598  0.059806
18              red onion     1  

## 1(c) Support and Confidence in item set mining

**Support**
- Support of an item set X is the fraction (or count) of transactions (recipes) that contain all items of X: `support(X) = (# recipes containing X) / (total # recipes)`.
- It measures how frequent / popular an item set is. An item set is *frequent* if its support is at least a user-defined minimum support threshold.
- Support is anti-monotone: a superset can never have higher support than any of its subsets. Apriori uses this to prune candidates.

**Confidence**
- For an association rule X → Y, confidence is the conditional probability of Y given X: `confidence(X → Y) = support(X ∪ Y) / support(X)`.
- It measures how reliable the rule is, i.e. among the recipes that contain X, the fraction that also contain Y.
- Unlike support, confidence is directional: confidence(X → Y) is generally not equal to confidence(Y → X).